# Lecture 5: Dynamic programming, the HJB equation and the linear-quadratic regulator

**The challenge.** The minimum principle gives *necessary* conditions. A
trajectory that satisfies them is an **extremal**, a candidate. Lecture 9 will
find two different extremals of the same cart-pole problem, with different
costs; there could be more. Which one is optimal? Is any of them? And what
*is* the costate, which Lecture 2 found numerically to be the sensitivity of
the cost to the state? Necessary conditions along one trajectory cannot
answer these questions, because they are about all trajectories at once.

This lecture takes the other road to optimal control, Bellman's, which
starts from the cost of *every* starting point. It answers both questions:
the costate is the gradient of the optimal cost, and a solution of the
Hamilton–Jacobi–Bellman equation certifies global optimality. It also shows
why this course nevertheless follows Pontryagin's road. And it solves the
one class of problems where both roads are easy, the linear-quadratic
regulator.

**After this lecture you can**

- state the principle of optimality and derive the HJB equation;
- show that $\lambda = \nabla V$ and recover the minimum principle from HJB;
- solve a linear-quadratic problem by the Riccati equation, and check that
  the extremal's costate is $\lambda = P x$;
- explain the curse of dimensionality, and what a local method gives up.

## 1. A name chosen to hide mathematics

Richard Bellman (1920–1984) worked at the RAND Corporation in Santa Monica
from 1952, on multistage decision problems for the Air Force. In his
autobiography, *Eye of the Hurricane* (1984), he explains where the name of
his method came from. The Secretary of Defense, Charles Wilson, "had a
pathological fear and hatred of the word *research*", and RAND was funded by
the Air Force, which answered to Wilson. Bellman needed a name for what he
was doing that no one could object to. "Programming" meant planning and
scheduling, and was respectable. "Dynamic" suggested time and change, and, he
wrote, it is impossible to use the word in a pejorative sense. So: *dynamic
programming*, "something not even a Congressman could object to".

The idea underneath is one sentence, the **principle of optimality**: *an
optimal policy has the property that, whatever the initial state and the
first decision, the remaining decisions are optimal for the state that
results.* The tail of an optimal trajectory is optimal. Bellman also coined
the phrase that describes its cost, the **curse of dimensionality**, which
section 5 explains.

In 1973 Bellman had a brain tumour removed. Complications from the surgery
left him severely disabled; he went on working and publishing until his death
in 1984.

## 2. The Hamilton–Jacobi–Bellman equation

Define the **value function**: the best cost achievable from state $x$ at
time $t$,
$$
V(t, x) = \min_{u(\cdot)} \Big[ \phi\big(t_f, x(t_f)\big) + \int_t^{t_f} L(s, x, u)\,ds \Big],
\qquad V(t_f, x) = \phi(t_f, x).
$$
By the principle of optimality, the best you can do from $(t, x)$ is to pick
a control for a short time $dt$, pay $L\,dt$, and then do the best from where
you land:
$$
V(t, x) = \min_{u} \Big[ L(t, x, u)\,dt + V\big(t + dt, x + f(t, x, u)\,dt\big) \Big].
$$
Expand $V$ to first order and let $dt \to 0$:
$$
\boxed{\; -\frac{\partial V}{\partial t} = \min_{u \in U} \Big[ L(t, x, u) + \nabla_x V^\top f(t, x, u) \Big] \;}
\qquad \text{(HJB)} .
$$
This is a partial differential equation for $V$ over the whole state space.
Compare the bracket with the Hamiltonian $H = L + \lambda^\top f$: they are
the same with $\lambda = \nabla_x V$. So along an optimal trajectory,

- **the costate is the gradient of the value function**, $\lambda(t) = \nabla_x V(t, x(t))$:
  the marginal cost of being somewhere else, exactly what Lecture 2 measured;
- the minimum in HJB is the **minimum principle**;
- differentiating HJB in $x$ along the trajectory gives
  $\dot\lambda = -H_x^\top$, the costate equation;
- $-V_t = H$, so with a free final time, where nothing is gained by waiting,
  $H = 0$.

The HJB equation descends from the equation Hamilton and Jacobi wrote for
mechanics in the 1830s (Lecture 3), through Constantin Carathéodory's "royal
road" to the calculus of variations (1935). Bellman's contribution was to see
it as a tool for computing decisions, and to give it the minimum over $u$.

**Sufficiency.** If a smooth $V$ solves HJB with the right terminal value,
the control that achieves the minimum is *globally* optimal (the
*verification theorem*). That is the certificate the necessary conditions
cannot give. The catch is that $V$ must be found everywhere.

## 3. The linear-quadratic regulator

There is one large class of problems where HJB can be solved in closed
form. Take linear dynamics and a quadratic cost,
$$
\dot x = A x + B u, \qquad
J = \tfrac12 x(T)^\top S_f\, x(T) + \int_0^T \tfrac12\big( x^\top Q x + u^\top R u \big)\,dt,
$$
with $Q, S_f \succeq 0$ and $R \succ 0$. Try $V = \tfrac12 x^\top P(t)\, x$.
The minimum over $u$ in HJB is at $u = -R^{-1} B^\top P x$, and HJB becomes
an ordinary differential equation for the matrix $P$, the **Riccati
equation**:
$$
-\dot P = A^\top P + P A - P B R^{-1} B^\top P + Q, \qquad P(T) = S_f .
$$
Integrate it backwards once, and the optimal control from *every* initial
state is the linear feedback $u = -R^{-1}B^\top P(t)\, x$. The costate is
$\lambda = \nabla V = P x$.

This is the **linear-quadratic regulator** (LQR), set out by Rudolf Kálmán in
1960 in "Contributions to the theory of optimal control", published in the
*Boletín de la Sociedad Matemática Mexicana* after a conference in Mexico
City. The same year Kálmán, born in Budapest in 1930 and then at RIAS in
Baltimore, published the filter that bears his name. Its reception was
sceptical enough that he placed it in a mechanical engineering journal, the
ASME *Journal of Basic Engineering*. Within a few years Stanley Schmidt's
group at NASA Ames had adapted it for the navigation of the Apollo
spacecraft. The two problems, estimation and control, turned out to be duals
of each other, with the same Riccati equation running in opposite directions
of time.

### The simplest case, again

Lecture 2's `soft_terminal_lq` is an LQR with $A = 0$, $B = 1$, $Q = 0$,
$R = 1$, $S_f = w$. The Riccati equation is $-\dot P = -P^2$, so
$P(t) = w / (1 + w(T - t))$, and $\lambda(0) = P(0)\,a = w a/(1 + wT)$, the
value Lecture 2 found by shooting and by finite differences. Now take a case
with some substance.

### A double integrator held near the origin

Steer $\ddot x = u$ from $(x, v) = (1, 0)$ for $T = 3$, penalizing position,
speed and effort along the way and the final state at the end:
$Q = \operatorname{diag}(1, 0.1)$, $R = 1$, $S_f = \operatorname{diag}(10, 10)$.
The final state is free; the terminal cost replaces a hard target.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp

from auto_optimal_control.homotopy import Homotopy
from auto_optimal_control.problem import OptimalControlProblem

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

t, x, v, u = sp.symbols("t x v u", real=True)
x0, v0 = sp.symbols("x_0 v_0", real=True)
lqr = OptimalControlProblem(
    name="double_integrator_lqr", time=t, states=[x, v], controls=[u], params=[x0, v0],
    dynamics=[v, u],
    running_cost=(x**2 + sp.Rational(1, 10) * v**2 + u**2) / 2,
    terminal_cost=5 * (x**2 + v**2),
    initial_state=[x0, v0], final_time=3,
)
solver = Homotopy(lqr)
r = solver.solve([1.0, 0.0], ([-5, -5], [5, 5]))
print(r.message, f"J = {r.objective:.10f}")

Now the other road: integrate the Riccati equation backwards from
$P(3) = S_f$ with SciPy, and compare the feedback law and $\lambda = P x$ with
the extremal the package computed.

In [ ]:
A = np.array([[0.0, 1.0], [0.0, 0.0]])
B = np.array([[0.0], [1.0]])
Q, Rm, Sf = np.diag([1.0, 0.1]), np.array([[1.0]]), np.diag([10.0, 10.0])


def riccati(s, p):
    P = p.reshape(2, 2)
    return -(A.T @ P + P @ A - P @ B @ np.linalg.solve(Rm, B.T) @ P + Q).ravel()


ric = solve_ivp(riccati, (3.0, 0.0), Sf.ravel(), dense_output=True, rtol=1e-12, atol=1e-12)
tr = r.shooting.trajectory(r.solution.v, n_points=301)
P = np.array([ric.sol(s).reshape(2, 2) for s in tr["t"]])
lam_riccati = np.einsum("kij,kj->ki", P, tr["z"][:, :2])
u_feedback = -np.einsum("ij,kjl,kl->ki", np.linalg.solve(Rm, B.T), P, tr["z"][:, :2])[:, 0]
print("max |lambda (extremal) - P x| =", np.abs(tr["z"][:, 2:] - lam_riccati).max())
print("max |u (extremal) - feedback| =", np.abs(tr["u"][:, 0] - u_feedback).max())
J_riccati = 0.5 * np.array([1.0, 0.0]) @ ric.sol(0.0).reshape(2, 2) @ np.array([1.0, 0.0])
print(f"V(0, x0) = x0' P(0) x0 / 2 = {J_riccati:.10f}")

The extremal's costate is $P x$ along the whole trajectory, its control is
the Riccati feedback, and its cost is the value function at the start. Both
roads arrive at the same place.

### Why LQ problems are easy for shooting too

For an LQ problem the canonical equations are *linear* in $(x, \lambda)$, so
the boundary value problem is a linear system: the shooting residual of
Lecture 9 is affine in the unknown initial costate. One Newton step from
*any* guess solves it exactly. The package exposes the residual and its
exact Jacobian directly:

In [ ]:
from auto_optimal_control.build import build_kernels
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.shooting import SingleShooting

kernels = build_kernels(derive(lqr))
shoot = SingleShooting(kernels, [1.0, 0.0])
guess = shoot.pack(lam0=[123.0, -45.0])          # any costate guess at all
R0, J0 = shoot.residual_jacobian(guess)
step = guess - np.linalg.solve(J0, R0)
print(f"residual before: {np.linalg.norm(R0):.1e}, after one Newton step: "
      f"{np.linalg.norm(shoot.residual(step)):.1e}")

## 4. The costate is a price

Section 2 says $\lambda(0) = \nabla_x V(0, x_0)$. The initial state is a
parameter of this problem, so we can test that directly: perturb $x_0$ and
$v_0$, re-solve, and difference the optimal cost.

In [ ]:
d = 1e-6
grad = []
for e in np.eye(2):
    Jp = solver.solve(np.array([1.0, 0.0]) + d * e, ([-5, -5], [5, 5])).objective
    Jm = solver.solve(np.array([1.0, 0.0]) - d * e, ([-5, -5], [5, 5])).objective
    grad.append((Jp - Jm) / (2 * d))
print("grad V(0, x0) by finite differences:", np.round(grad, 8))
print("lambda(0) of the extremal:          ", np.round(r.solution.lam0, 8))

This interpretation is useful well beyond LQ problems. A costate component
that is large says the cost is sensitive to that state; one that is zero says
the state does not matter at that moment (a free final state has
$\lambda(t_f) = 0$ for exactly that reason). In economics, where this theory
is also used, costates are called **shadow prices**.

## 5. The curse of dimensionality, and the road this course takes

If HJB gives global optimality and feedback, why not always solve it?
Because $V$ is a function on the whole state space. On a grid of $N$ points
per state, $n$ states take $N^n$ points, at every time step:

In [ ]:
for n, name in [(2, "double integrator"), (4, "cart-pole"), (6, "robot arm"),
                (7, "spacecraft (position, velocity, mass)"), (12, "6-joint arm with velocities")]:
    print(f"n = {n:2d} {name:38s} 100^n = {100.0**n:9.1e} grid points")

Bellman knew this, and named it. Except in special cases like LQR, dynamic
programming is practical for a handful of states. Pontryagin's conditions
cost the opposite: they live on *one* trajectory, an ODE in $2n$ variables,
whatever the dimension. The price is that they are **local** and
**necessary**: they find extremals, not certificates. That is the road of
this course, and its consequences are concrete:

- An extremal can be a local minimum, a saddle, or even a maximum. The
  classical second-order tests (Legendre's condition, 1786, that $H_{uu}$ be
  positive; Jacobi's 1837 condition that no *conjugate point* occur) rule
  out some of these, locally. Lecture 12 meets a version of Legendre's
  condition, Legendre–Clebsch, for singular arcs.
- Different starting guesses can converge to different extremals. The only
  practical way to rank them is by cost, and the only way to find more of
  them is to search. Lecture 9 does that for the cart-pole.

Some problems have a *convex* structure that makes the necessary conditions
sufficient, LQR among them; then any extremal is the global optimum. In
general they do not, and a careful user treats every converged extremal as
"the best found so far".

## Summary

- The value function $V(t, x)$ is the optimal cost-to-go. The principle of
  optimality gives the HJB equation $-V_t = \min_u [L + \nabla V^\top f]$.
- The costate is $\nabla V$ along the extremal; the minimum principle and the
  costate equation follow from HJB. A smooth solution of HJB certifies global
  optimality.
- For linear dynamics and quadratic cost, $V = \tfrac12 x^\top P x$ with $P$
  from the Riccati equation, the optimal control is linear feedback, and
  $\lambda = P x$; shooting solves the linear BVP in one Newton step.
- HJB suffers the curse of dimensionality; the indirect method does not, at
  the price of being local and necessary only.

## Exercises

1. For the double integrator LQR, drop the terminal cost and impose
   $x(3) = v(3) = 0$ instead (`final_state`). Does $\lambda = P x$ still hold
   for some $P$? (Hint: the terminal condition for the Riccati equation
   becomes infinite; integrate the equation for $P^{-1}$ instead.)
2. Make $Q$ larger by factors of 10. How does the closed-loop response
   change, and how does $\lambda(0)$ change? Interpret $\lambda$ as a price.
3. Solve HJB numerically for the scalar problem $\dot x = u$,
   $\int (x^2 + u^2)/2$, on a grid in $x$, by stepping
   $V(t - dt, x) = \min_u [L\,dt + V(t, x + u\,dt)]$ backwards. Compare with
   the Riccati solution. How does the cost of your code scale with the number
   of states?

## Further reading

- R. Bellman, *Dynamic Programming* (Princeton, 1957), and *Eye of the
  Hurricane: an Autobiography* (World Scientific, 1984), on the name.
- R. E. Kalman, "Contributions to the theory of optimal control", *Boletín de
  la Sociedad Matemática Mexicana* 5, 1960.
- D. E. Kirk, *Optimal Control Theory: An Introduction* (1970), chapters 3
  and 5.
- D. P. Bertsekas, *Dynamic Programming and Optimal Control*, vol. 1, chapter
  3: HJB and its relation to the minimum principle.